<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

# 학습 내용
>이번 장에서는 <strong>RAG 파이프라인(RAG Pipeline)</strong>에 대해 학습합니다.
>LangGraph StateGraph로 Retrieve → Generate 파이프라인을 학습해봅시다.

# RAG (Retrieval-Augmented Generation)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">검색(Retrieval)</mark>으로 관련 문서를 찾고, 이를 컨텍스트로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">생성(Generation)</mark>하는 파이프라인입니다.

<mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">RAG(Retrieval-Augmented Generation, 검색 증강 생성)</mark>는 LLM의 할루시네이션 문제를 외부 지식으로 보완하는 아키텍처입니다. 핵심 아이디어는 단순합니다. LLM에게 "이 자료를 보고 답해"라고 알려주면, LLM은 자신이 학습한 내용 대신 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">제공된 자료에 근거해 답변</mark>합니다. 이렇게 하면 최신 정보, 조직 내부 문서, 수시로 바뀌는 데이터도 LLM이 정확히 다룰 수 있습니다.</br>

RAG 파이프라인의 전체 흐름은 다음과 같습니다. (1) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">문서 로드</mark>로 PDF, 웹 페이지 등 원본 문서를 `Document` 객체로 변환하고, (2) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">청킹</mark>으로 긴 문서를 토큰 제한에 맞는 작은 조각으로 분할합니다. (3) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">임베딩</mark>으로 각 청크를 고차원 벡터로 변환한 뒤, (4) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">벡터 저장</mark>으로 변환된 벡터를 벡터 데이터베이스(예: Chroma)에 저장합니다. (5) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">검색(Retrieve)</mark>에서 사용자 질문을 벡터로 변환하여 가장 유사한 청크를 검색하고, (6) <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">생성(Generate)</mark>에서 검색된 청크를 컨텍스트로 LLM에 전달하여 최종 답변을 생성합니다. 이 장에서는 5번과 6번, 즉 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">검색과 생성 단계를 LangGraph로 연결하는 방법</mark>을 학습합니다.</br>

이 과정에서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">키워드/의미 검색</mark>(문서에서 관련 청크를 찾는 두 가지 방식, Ch.4-1_003, Ch.4-1_004 참고), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">벡터 데이터베이스</mark>(임베딩 벡터를 저장하고 유사도 검색을 수행하는 DB), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">청킹</mark>(문서를 검색 가능한 크기로 분할하는 전처리 단계, Ch.4-1_005 참고), 그리고 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM API</mark>(프롬프트를 전송하고 텍스트 응답을 받는 기본 호출 패턴)에 대한 이해가 필요합니다.

## RAGState 정의
> StateGraph의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">상태 타입</mark>을 TypedDict로 정의합니다.

In [ ]:
# TODO 1: 상태 타입을 정의하여 question(str), context(List[Document]), answer(str) 필드를 선언한 뒤 필드 목록을 출력해봅시다.

from typing import TypedDict, List
from langchain_core.documents import Document

class RAGState(TypedDict):
    question: str
    context: List[Document]
    answer: str

print(f"RAGState 필드: {list(RAGState.__annotations__.keys())}")

## 노드 함수 구현
> 각 노드는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">상태를 받아 상태를 반환</mark>하는 함수입니다.

In [ ]:
# TODO 2: retrieve 함수(검색기로 관련 문서를 검색)와 generate 함수(프롬프트 템플릿과 LLM으로 답변 생성)를 RAGState를 받아 반환하도록 구현해봅시다.

def retrieve(state: RAGState) -> RAGState:
    """질문으로 관련 문서 검색"""
    documents = retriever.invoke(state["question"])
    return {"context": documents}

def generate(state: RAGState) -> RAGState:
    """검색된 문서를 기반으로 답변 생성"""
    context = "\n".join([doc.page_content for doc in state["context"]])
    prompt = ChatPromptTemplate.from_template("""
    다음 자료를 기반으로 답변하세요.
    자료: {context}
    질문: {question}
    """)
    chain = prompt | llm
    response = chain.invoke({
        "context": context,
        "question": state["question"]
    })
    return {"answer": response.content}

## StateGraph 조립
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">노드를 추가하고 엣지로 연결</mark>하여 그래프를 완성합니다.

In [ ]:
# TODO 3: 상태 그래프를 생성하고, retrieve와 generate 노드를 추가한 뒤 START→retrieve→generate→END로 엣지를 연결하여 컴파일 후 "매출 현황은?"으로 실행해봅시다.

from langgraph.graph import StateGraph, START, END

graph = StateGraph(RAGState)

# 노드 추가
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

# 엣지 연결
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

# 컴파일 및 실행
app = graph.compile()
result = app.invoke({"question": "매출 현황은?"})
print(f"질문: {result['question']}")
print(f"검색 문서: {len(result['context'])}건")
print(f"답변: {result['answer']}")

## RAG 파이프라인 구성요소 정리

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">구성요소</th>
      <th>역할</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>RAGState</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">상태 스키마</mark> 정의</td></tr>
    <tr><td style="text-align:center"><code>retrieve</code></td><td>벡터 검색 → context 업데이트</td></tr>
    <tr><td style="text-align:center"><code>generate</code></td><td>context 기반 → answer 생성</td></tr>
    <tr><td style="text-align:center"><code>StateGraph</code></td><td>노드와 엣지로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">워크플로우 정의</mark></td></tr>
    <tr><td style="text-align:center"><code>compile()</code></td><td>실행 가능한 앱으로 변환</td></tr>
  </tbody>
</table>

💡StateGraph의 장점
> 단순 체인(`prompt | llm`)보다 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">분기, 반복, 조건부 실행</mark>이 가능합니다.
> Agent 확장 시 동일한 패턴으로 복잡한 워크플로우를 구성할 수 있습니다.

💡RAG vs Fine-tuning
> RAG: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">외부 지식을 실시간</mark>으로 주입 (비용 낮음, 최신성 보장)
> Fine-tuning: 모델 가중치 자체를 변경 (비용 높음, 도메인 특화)